In [ ]:
# 1. Install dependencies
!pip install ultralytics -q

# 2. Upload your model and test image
from google.colab import files
print("Upload your YOLO model (.pt file)")
model_upload = files.upload()  # upload your .pt file

print("Upload your test image(s)")
image_upload = files.upload()  # upload one or more images

In [ ]:
# 3. Run inference and visualize
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import os

# Load your model
model_name = list(model_upload.keys())[0]
model = YOLO(model_name)

# Run on all uploaded images
image_names = list(image_upload.keys())

for img_name in image_names:
    results = model(img_name, conf=0.25)  # adjust conf threshold as needed
    
    # Plot with bounding boxes
    annotated = results[0].plot()  # returns BGR numpy array
    annotated_rgb = annotated[:, :, ::-1]  # BGR → RGB
    
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_rgb)
    plt.title(f"Detections: {img_name}")
    plt.axis("off")
    plt.show()
    
    # Print detection details
    boxes = results[0].boxes
    print(f"\n--- {img_name} ---")
    print(f"Total detections: {len(boxes)}")
    for box in boxes:
        cls_id = int(box.cls)
        conf = float(box.conf)
        xyxy = box.xyxy[0].tolist()
        label = model.names[cls_id]
        print(f"  [{label}] conf={conf:.2f}  box={[round(x,1) for x in xyxy]}")

In [ ]:
# 4. (Optional) Save annotated images to disk and download
save_dir = "results"
os.makedirs(save_dir, exist_ok=True)

for img_name in image_names:
    results = model(img_name, conf=0.25)
    annotated = results[0].plot()
    out_path = os.path.join(save_dir, f"detected_{img_name}")
    Image.fromarray(annotated[:, :, ::-1]).save(out_path)
    print(f"Saved: {out_path}")

# Download all results
for f in os.listdir(save_dir):
    files.download(os.path.join(save_dir, f))